<a href="https://colab.research.google.com/github/zhengyangsu/Natural-Language-Processing/blob/main/latin_translation_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Latin to English Translation with LoRA

Adapted from the Week 1 tutorial `movie_review_sentiment_classifier_with_bert.ipynb`  
Use a GPU runtime  
Fine-tunes a Latin-to-English translation model with QLoRA (4-bit base model, small trainable LoRA adapters).

# 1. Setup Python Libraries (pip)
Installs the libraries, then imports everything the notebook needs in one place.

In [21]:
# transformers = models and tokenizers, peft = LoRA, bitsandbytes = 4-bit loading,
# accelerate = GPU placement, datasets = data loading, --quiet hides the long install log
!pip install -q transformers peft bitsandbytes accelerate datasets tensorboard langid sacrebleu evaluate

# --- all imports in one place ---
import gc, json, math, os, re, time, unicodedata, urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
from IPython.display import display

import evaluate
import transformers, peft
from datasets import load_dataset
from google.colab import drive
from huggingface_hub import notebook_login, whoami #Will speed up download speed for Huggingface
from langid.langid import LanguageIdentifier, model as langid_model # langid to check latin in en key
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments, pipeline)

# 2. Mount Google Drive
Create working directories and download custom datasets

In [22]:
drive.mount('/content/drive')   # skip if already mounted

# Path to NLP Project folder
project_path = '/content/drive/MyDrive/NLP_Project'

# Create directory if does not exist
if not os.path.exists(project_path):
    os.makedirs(project_path)
    print(f"Created dir: {project_path}")

# Change working directory
os.chdir(project_path)
print(f"Current working dir set to: {os.getcwd()}")

# Create the dataset folder (does nothing if it already exists)
os.makedirs('dataset', exist_ok=True)

# Download the custom test set
URL = 'https://raw.githubusercontent.com/zhengyangsu/Natural-Language-Processing/main/data/plautus_amphitruo_test50.jsonl'
DEST = 'dataset/plautus_amphitruo_test50.jsonl'
urllib.request.urlretrieve(URL, DEST)

# Check the download
print(os.listdir('dataset'))
with open(DEST, encoding='utf8') as f:
    print(sum(1 for _ in f), 'lines')   # should print 50


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current working dir set to: /content/drive/MyDrive/NLP_Project
['plautus_amphitruo_test50.jsonl']
50 lines


# 3. Hugging Face Auth
Logs in to Hugging Face for authenticated, faster downloads

In [23]:
#!hf auth login
#!hf auth whoami
notebook_login()
print(whoami()['name'])

zhengyangsu


# 4. Load the Dataset
Loads the Latin-English parallel dataset and the custom Perseus test set, and prints the official split sizes.

In [24]:
raw = load_dataset('grosenthal/latin_english_parallel')
mine = load_dataset('json', data_files={'test': 'dataset/plautus_amphitruo_test50.jsonl'})['test']

Generating test split: 0 examples [00:00, ? examples/s]

In [25]:
# Official raw split sizes check, before processing---
total = sum(len(v) for v in raw.values())
print(f"{'split':6s} {'rows':>8s} {'share':>7s}")
for k, v in raw.items():
    print(f'{k:6s} {len(v):>8,} {len(v) / total:>7.1%}')
print(f"{'total':6s} {total:>8,}")
print('columns:', raw['train'].column_names)

split      rows   share
train    99,343   98.0%
test      1,014    1.0%
valid     1,014    1.0%
total   101,371
columns: ['id', 'la', 'en', 'file']


## Data Cleanup
Normalizes text, strips HTML tags, and drops short sentences, rows whose English column is actually Latin, and misaligned pairs, logging the count for each drop.

In [26]:
ident = LanguageIdentifier.from_modelstring(langid_model, norm_probs=True)
drop_counts = Counter()   # check counter


# Unicode normalize, remove html tags, collapse space
def clean_text(s):
    s = unicodedata.normalize('NFC', s)       # unify Unicode forms
    s = re.sub(r'</?[a-zA-Z][^>]*>', ' ', s)  # remove HTML tags
    return ' '.join(s.split())                # collapse whitespace

# True if en column is 0.9 confident detected as Latin
def en_is_latin(x):
    if len(x['en'].split()) < 6:          # skip short text
        return False
    lang, prob = ident.classify(x['en'])
    return lang == 'la' and prob > 0.9

# Keep pair only if it passes all checks
def is_ok(x):
    n_la, n_en = len(x['la'].split()), len(x['en'].split())
    if n_la < 3 or n_en < 3:              # sentence too short
        drop_counts['short_sentence'] += 1
        return False
    if en_is_latin(x):                    # en column is latin
        drop_counts['en_is_latin'] += 1
        return False
    if not (0.3 <= n_en / n_la <= 4):     # lengths misalignment
        drop_counts['length_misalignment'] += 1
        return False
    return True

# Text cleaning
raw = raw.map(lambda x: {'la': clean_text(x['la']), 'en': clean_text(x['en'])})

# Filter splits
for k in ['train', 'valid', 'test']:
    drop_counts.clear() # reset split filter count
    before = len(raw[k])
    raw[k] = raw[k].filter(is_ok, load_from_cache_file=False)   # force counters update
    print(f"\n[{k}] {before} -> {len(raw[k])}")
    for reason in ['short_sentence', 'en_is_latin', 'length_misalignment']:
        print(f"  {reason:20s}: {drop_counts[reason]}")
    print(f"  {'total dropped':20s}: {sum(drop_counts.values())}")

Filter: 100%|##########| 99343/99343 [00:00<?, ? examples/s]


[train] 99343 -> 98225
  short_sentence      : 353
  en_is_latin         : 736
  length_misalignment : 29
  total dropped       : 1118


Filter: 100%|##########| 1014/1014 [00:00<?, ? examples/s]


[valid] 1014 -> 1002
  short_sentence      : 1
  en_is_latin         : 10
  length_misalignment : 1
  total dropped       : 12


Filter: 100%|##########| 1014/1014 [00:00<?, ? examples/s]


[test] 1014 -> 996
  short_sentence      : 5
  en_is_latin         : 12
  length_misalignment : 1
  total dropped       : 18


## Add EOS and Cap MAX_LEN
Defines the Latin: ... English: ... prompt format with an end of sequence token, removes examples longer than 256 tokens.

In [27]:
checkpoint = 'hathibelagal/llama-3.2-latin'  #base model
tokenizer = AutoTokenizer.from_pretrained(checkpoint, clean_up_tokenization_spaces=False)
tokenizer.pad_token = tokenizer.eos_token

MAX_LEN = 256
EOS = tokenizer.eos_token

def build_text(la, en):
    return f"Latin: {la}\nEnglish: {en}{EOS}"           # add EOS

# remove beyond MAX_LEN
def fits(batch):
    texts = [build_text(la, en) for la, en in zip(batch['la'], batch['en'])]
    return [len(ids) <= MAX_LEN for ids in tokenizer(texts)['input_ids']]

for k in ['train', 'valid', 'test']:
    before = len(raw[k])
    raw[k] = raw[k].filter(fits, batched=True, batch_size=1000)
    after = len(raw[k])
    n_dropped = before - after
    print(f"{k}: {before} -> {after} | dropped {n_dropped} ({n_dropped / before * 100:.2f}%)")

train: 98225 -> 96455 | dropped 1770 (1.80%)
valid: 1002 -> 991 | dropped 11 (1.10%)
test: 996 -> 974 | dropped 22 (2.21%)


## Check for Overlapping
Builds train (from percentages), validation (fixed 900) and test sets (fixed 900), removes duplicates and any test or validation sentences found in train, and verifies there is no overlap.

In [28]:
# --- split sizes as percentages ---
TRAIN_PCT = 10     # % of train set
SEED      = 42

N_TRAIN = round(len(raw['train']) * TRAIN_PCT / 100)
N_VAL   = 900
N_TEST  = 900
print(f"requested: train {N_TRAIN} ({TRAIN_PCT}% of {len(raw['train'])}) | val {N_VAL} | test {N_TEST}")


# Only used for comparing sentences, never changes the data
def key(s):
    return s.lower()

# 1. Check split overlapping
full_train_la = {key(x) for x in raw['train']['la']}
print('test/train overlap:', sum(key(x) in full_train_la for x in raw['test']['la']), '/', len(raw['test']))
print('valid/train overlap:', sum(key(x) in full_train_la for x in raw['valid']['la']), '/', len(raw['valid']))

# val/test set size
val = raw['valid'].select(range(N_VAL))
val_la = {key(x) for x in val['la']}
# take test rows in val
test = raw['test'].filter(lambda x: key(x['la']) not in val_la).select(range(N_TEST))

# remove val/test sentences from train
held_la = {key(x) for x in val['la']} | {key(x) for x in test['la']}
held_en = {key(x) for x in val['en']} | {key(x) for x in test['en']}

pool = raw['train'].filter(
    lambda x: key(x['la']) not in held_la and key(x['en']) not in held_en
)
print('train pool:', len(raw['train']), '->', len(pool))

# remove duplicate Latin sentences inside train
seen = set()
def first_time(x):
    k = key(x['la'])
    if k in seen:
        return False
    seen.add(k)
    return True
pool = pool.filter(first_time)
print('unique:', len(pool))

dataset = {
    'train': pool.shuffle(seed=SEED).select(range(N_TRAIN)),
    'val':   val,
    'test':  test,
}

# verification
train_la = {key(x) for x in dataset['train']['la']}
print('train ∩ test:', sum(key(x) in train_la for x in dataset['test']['la']))
print('train ∩ val :', sum(key(x) in train_la for x in dataset['val']['la']))
print('val ∩ test  :', len({key(x) for x in dataset['val']['la']} & {key(x) for x in dataset['test']['la']}))

requested: train 9646 (10% of 96455) | val 900 | test 900
test/train overlap: 7 / 974
valid/train overlap: 6 / 991
train pool: 96455 -> 96343
unique: 96109
train ∩ test: 0
train ∩ val : 0
val ∩ test  : 0


##Dataframe Display
Shows few rows of each split to check the data

In [29]:
print({k: len(v) for k, v in raw.items()})
print(dataset['train'].column_names)

pd.set_option('display.max_colwidth', None)

dataset['train'].to_pandas().head()


for name in ['train', 'val', 'test']:
    print(name)
    display(dataset[name].to_pandas().head(3))

{'train': 96455, 'test': 974, 'valid': 991}
['id', 'la', 'en', 'file']
train


,id,la,en,file
0,37954,hunc miseratus adest infesto vulnere Ladmus.,"In pity for him Ladmus came up, intent to strike.",final_alignments\Silius_Italicus_Punica_Book1.json
1,68502,in duobus contristatum est cor meum et in tertio iracundia mihi advenit,"At two things my heart is grieved, and the third bringeth anger upon me.",final_alignments\Vulgate_Bible.json
2,13882,Tertiam autem aciem in sinistrum suum cornu contulerat et usque ad aciei suae mediam legionem porrexerat et ita collocaverat uti sinistrum suum cornu esset triplex.,"His focus on the third line was directed towards his left wing which he extended all the way to the central legion of his line, organizing it in a way that made his left wing three lines in total.",final_alignments\Caesar_African.json


val


,id,la,en,file
0,36149,"nam memor originis tuae nobilium sibi persuasit universitas, quamdiu nos Sabini familia rexerit, Sabiniani familiam non timendam.","For the entire aristocracy, who recall the lineage that you belong to, are certain that as long as the descendants of Sabinus are our leaders, there is no reason for us to be afraid of the descendants of Sabinianus.",final_alignments\Sidonius_Letters_Book3.json
1,27252,"Tanta tibi bene faciendi vis, ut indulgentia tua necessitates aemuletur.",Such is your ability to perform well that your merciful kindness has an equivalent impact as harsh compulsion.,final_alignments\Pliny_Younger_Panegyricus.json
2,28719,"tempore nec senior Pater est, nec numine maior, nam sapiens retro semper Deus edidit ex se, per quod semper erat, gignenda ad saecula Verbum.","The Father is not older in time nor greater in divinity because God has been wise throughout infinite time and released from Himself the Word to create the world, through which he always existed.",final_alignments\Prudentius_Divinity_Of_Christ.json


test


,id,la,en,file
0,32668,"Tibi autem, qui sapis, quam potest denuntio ipsi mihi indices quid ex eis cognoveris, ne possis infitiari dixisse, quae dixeris.","But I offer you a serious caution that you tell me directly what you have gained from those sacrifices, since I know, so that you will not be able to deny having spoken to me.",final_alignments\Quintus_Curtius_Alexander_Book7.json
1,90310,currens autem vir de Beniamin ex acie venit in Silo in die illo scissa veste et conspersus pulvere caput,"And there ran a man of Benjamin out of the army, and came to Silo the same day, with his clothes rent, and his head strewed with dust.",final_alignments\Vulgate_Bible.json
2,16191,"Exposuit quid iniquitas loci posset, quid ipse ad Avaricum sensisset, cum sine duce et sine equitatu deprehensis hostibus exploratam victoriam dimisisset, ne parvum modo detrimentum in contentione propter iniquitatem loci accideret.","""And he said what disadvantage the ground could have on the outcome, recalling his own opinion at Avaricum where, despite having surprised the enemy without the aid of a general or cavalry, he abandoned a certain victory in order to avoid even a small loss caused by the disadvantage of the position.""",final_alignments\Caesar_DBG_Book7.json


# Tokenize the Dataset
Converts text to token ids and records the prompt length, so the loss can ignore the Latin input. Checks that every example ends with EOS.

In [30]:
def tokenization(batch):
    texts   = [build_text(la, en) for la, en in zip(batch['la'], batch['en'])]
    prompts = [f"Latin: {la}\nEnglish:" for la in batch['la']]                  # new: the part to ignore in the loss
    enc = tokenizer(texts, truncation=True, max_length=MAX_LEN)
    enc['prompt_len'] = [len(ids) for ids in tokenizer(prompts)['input_ids']]    # new: how many tokens the prompt takes
    return enc

tokenized = {k: v.map(tokenization, batched=True, remove_columns=v.column_names)
             for k, v in dataset.items()}

assert all(ids[-1] == tokenizer.eos_token_id for ids in tokenized['train']['input_ids'][:100]), 'EOS is missing'


# Load the Model and Add LoRA
LoRa configurations
Loads the 3B base model in 4-bit and attaches LoRA adapters to the chosen layers. LABEL foe each experiment.

In [31]:
LABEL = 'default_qv'   # name of experiment, change it every train

# --- clear weight from previous run (safe on the first run too) ---
for name in ['trainer', 'model', 'base', 'pipe']:     # anything holding the old model
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

# --- fresh base model, no LoRA weights ---

bnb = BitsAndBytesConfig(load_in_4bit=True,  # frozen base weights in 4 bits
                         bnb_4bit_quant_type='nf4',  # NF4, 4-bit format for QLoRA
                         bnb_4bit_compute_dtype=torch.float16)  # compute in 16-bit floats

model = AutoModelForCausalLM.from_pretrained(checkpoint,  # download base model
                                             quantization_config=bnb,  # apply 4-bit settings
                                             device_map='auto')  # training on GPU

model = prepare_model_for_kbit_training(
    model,
    gradient_checkpointing_kwargs={'use_reentrant': False})  # casts some layers to full precision, cant update 4bit weight


'''
r=8 and lora_alpha=8 (scale 1.0)
lora_dropout=0.0
target_modules=None default q_proj and v_proj only

model = get_peft_model(model, LoraConfig(r=16,  # rank of the LoRA matrices: higher = more capacity
                                         lora_alpha=32,  # scaling factor for LoRA update
                                         target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',      # attention
                                         'gate_proj', 'up_proj', 'down_proj'],            # MLP
                                         task_type='CAUSAL_LM'))  # decoder-only, left to right prediction

'''

#Remove target_modules for your training, add your hyperparameters
model = get_peft_model(model, LoraConfig(target_modules=
                                          ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                          'gate_proj', 'up_proj', 'down_proj'],
                                          task_type='CAUSAL_LM'))  # decoder-only, left to right prediction
model.print_trainable_parameters()  # show how few parameters are actually trained

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 12,156,928 || all params: 3,224,906,752 || trainable%: 0.3770


# Setup Training Configurations
Training argument settings
Defines the loss masking collator, the default training hyperparameters, logging and checkpointing, and builds the Trainer

In [32]:
N_EPOCH = 3  # Training epochs


class CausalLMCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        self.tokenizer.padding_side = 'right'
        prompt_lens = [f.pop('prompt_len') for f in features]      # take it out before padding
        batch = self.tokenizer.pad(features, return_tensors='pt')
        labels = batch['input_ids'].clone()
        labels[batch['attention_mask'] == 0] = -100                 # ignore padding
        for i, n in enumerate(prompt_lens):                         # ignore the Latin prompt
            labels[i, :n] = -100
        batch['labels'] = labels
        return batch

# Warmup as an integer number of steps (3% of training); works across transformers versions
steps_per_epoch = math.ceil(len(tokenized['train']) / (4 * 4))      # batch 4 x grad_accum 4
total_steps = steps_per_epoch * N_EPOCH
WARMUP_STEPS = max(1, round(0.03 * total_steps))
print('total steps:', total_steps, '| warmup steps:', WARMUP_STEPS)

training_args = TrainingArguments(
    seed=SEED,
    output_dir=f'./results_{LABEL}',
    num_train_epochs=N_EPOCH,  # total passes
    per_device_train_batch_size=4,  # 4 examples/step (small to fit GPU memory)
    gradient_accumulation_steps=4,  # accumulate 4 steps before each update
    learning_rate=2e-4,  # step size
    lr_scheduler_type='cosine',  # smooth decay of the learning rate
    warmup_steps=WARMUP_STEPS,  # ramp the learning rate up over the first 3% of steps
    optim='paged_adamw_8bit',  # 8-bit optimizer states, saves GPU memory
    max_grad_norm=0.3,  # clip gradients (value used in the QLoRA paper)
    weight_decay=0.0,  # no weight decay on the LoRA weights
    fp16=True,  # 16-bit training (T4 supports fp16 not bf16)
    eval_strategy='epoch',  # evaluate on the validation set of each epoch
    save_strategy='epoch',
    save_total_limit=2,  # keep the 2 newest checkpoints
    load_best_model_at_end=True,  # load the best checkpoint after training
    greater_is_better=False,
    logging_strategy='steps',
    logging_steps=50,
    train_sampling_strategy='group_by_length',  # batch similar length sentences together to cut padding
    remove_unused_columns=False,  # keep prompt_len for the collator
    report_to='tensorboard',
)

trainer = Trainer(
    model=model,  # LoRA model
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['val'],
    data_collator=CausalLMCollator(tokenizer)
)

# Sanity check: show model is trained on English part only
batch = CausalLMCollator(tokenizer)([dict(tokenized['train'][0])])
print(tokenizer.decode(batch['input_ids'][0]))
print('trained on:', tokenizer.decode([t for t in batch['labels'][0] if t != -100]))

total steps: 1809 | warmup steps: 54
<|begin_of_text|>Latin: hunc miseratus adest infesto vulnere Ladmus.
English: In pity for him Ladmus came up, intent to strike.<|end_of_text|>
trained on:  In pity for him Ladmus came up, intent to strike.<|end_of_text|>


# Define evaluation
Defines translation and scoring: test loss, perplexity (original test set only), BLEU and chrF on the test set and the Perseus set.

In [33]:
# Metric scorers: BLEU and chrF
# Load the translation metrics
bleu  = evaluate.load('sacrebleu')   # word n-gram overlap with the reference
chrf  = evaluate.load('chrf')        # character n-gram overlap

def translate_batch(latin_list, max_new_tokens=120):
    """Translate a list of Latin sentences in one batch and return the English outputs."""
    tokenizer.padding_side = 'left'   # pad on the left
    try:
        prompts = [f"Latin: {la}\nEnglish:" for la in latin_list]

        # Convert text to token ids; pad to equal length and cap the prompt at MAX_LEN tokens
        inputs = tokenizer(prompts, return_tensors='pt', padding=True,
                           truncation=True, max_length=MAX_LEN).to(model.device)

        with torch.no_grad():   # no gradients needed for inference
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False,                       # greedy decoding
                                 pad_token_id=tokenizer.eos_token_id)   # avoid pad_token_id warning

        # Drop the prompt tokens and decode only the newly generated part
        texts = tokenizer.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        # Keep only the first line, in case the model keeps writing after the translation
        return [t.split('\n')[0].strip() for t in texts]
    finally:
        tokenizer.padding_side = 'right'   # restore right padding for training

def full_eval(label, batch_size=32):
    """Report train/val/test loss, perplexity, BLEU and chrF, and save predictions. `label` names the output files."""
    torch.cuda.empty_cache()
    model.eval()   # switch off dropout and other training-only behavior
    print(f'===== {label} =====')

    # 1. Loss on train / val / test, all measured the same way (eval mode, current weights)
    #    The 'eval_<name>' prefix names the table column "Train Loss" / "Val Loss" / "Test Loss"
    #    and keeps these results out of the validation curve (which uses the exact key 'eval_loss')
    loss_sets = {
        'train': tokenized['train'].select(range(900)),   # 900-sentence sample of the (already shuffled) train set
        'val':   tokenized['val'],
        'test':  tokenized['test'],
    }
    scores = {}
    for name, ds in loss_sets.items():
        r = trainer.evaluate(ds, metric_key_prefix=f'eval_{name}')
        scores[f'{name}_loss'] = r[f'eval_{name}_loss']
    print(' | '.join(f"{k} {v:.4f}" for k, v in scores.items()),
          f"| test perplexity {np.exp(scores['test_loss']):.2f}")

    # 2. Translate each test set, then score the translations against the references
    for name, split in [('test', dataset['test']), ('mine', mine)]:
        preds = []
        for i in tqdm(range(0, len(split), batch_size), desc=name):
            preds += translate_batch(split['la'][i:i + batch_size])   # translate one batch at a time

        refs_nested = [[r] for r in split['en']]   # sacrebleu / chrF expect a list of reference lists

        b = bleu.compute(predictions=preds, references=refs_nested)['score']
        c = chrf.compute(predictions=preds, references=refs_nested)['score']

        print(f"{name:5s} | BLEU {b:.2f} | chrF {c:.2f}")
        scores[f'{name}_bleu'] = b
        scores[f'{name}_chrf'] = c

        # Save Latin/reference/prediction for error analysis
        with open(f'{run_dir(label)}/preds_{name}.jsonl', 'w', encoding='utf-8') as f:
            for la, ref, pred in zip(split['la'], split['en'], preds):
                f.write(json.dumps({'la': la, 'ref': ref, 'pred': pred}, ensure_ascii=False) + '\n')

    return scores   # e.g. results['attn_only'] = full_eval('attn_only')

##Logging
Helper that save each run's metrics, config, loss curves and adapter into runs label

In [34]:
def run_dir(label):
    """Folder that holds everything for one experiment: runs/<label>/"""
    d = f'runs/{label}'
    os.makedirs(d, exist_ok=True)
    return d

def _xy(rows):
    return tuple(zip(*rows)) if rows else ([], [])

def save_metrics(label, scores, extra=None):
    path = f'{run_dir(label)}/metrics.json'
    with open(path, 'w') as f:
        json.dump({**scores, **(extra or {})}, f, indent=2, default=str)
    print('saved', path)

def save_training(label):
    d = run_dir(label)
    logs = trainer.state.log_history
    json.dump(logs, open(f'{d}/log_history.json', 'w'))

    cfg = {
        'label': label,
        'lora': model.peft_config['default'].to_dict(),
        'trainable_params': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'training_args': training_args.to_dict(),
        'dataset_sizes': {k: len(v) for k, v in dataset.items()},
        'versions': {'transformers': transformers.__version__, 'peft': peft.__version__, 'torch': torch.__version__},
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    }
    json.dump(cfg, open(f'{d}/config.json', 'w'), indent=2,
              default=lambda o: sorted(o) if isinstance(o, set) else str(o))

    tr = [(l['step'], l['loss']) for l in logs if 'loss' in l]
    ev = [(l['step'], l['eval_loss']) for l in logs if 'eval_loss' in l]
    lr = [(l['step'], l['learning_rate']) for l in logs if 'learning_rate' in l]
    gn = [(l['step'], l['grad_norm']) for l in logs if 'grad_norm' in l]
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(*_xy(tr), label='train (avg per 50 steps)')
    ax[0].plot(*_xy(ev), 'o-', label='validation (per epoch)')
    ax[0].set_title(f'Loss - {label}'); ax[0].set_xlabel('step'); ax[0].legend()
    ax[1].plot(*_xy(lr)); ax[1].set_title('Learning rate'); ax[1].set_xlabel('step')
    ax[2].plot(*_xy(gn)); ax[2].set_title('Gradient norm'); ax[2].set_xlabel('step')
    plt.tight_layout()
    plt.savefig(f'{d}/loss_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

    model.save_pretrained(f'{d}/adapter')
    print('saved to', d, '->', sorted(os.listdir(d)))

def compare_runs():
    rows, hist = {}, {}
    for label in sorted(os.listdir('runs')):
        m, h = f'runs/{label}/metrics.json', f'runs/{label}/log_history.json'
        if os.path.exists(m): rows[label] = json.load(open(m))
        if os.path.exists(h): hist[label] = json.load(open(h))

    df = pd.DataFrame(rows).T

    cols = [c for c in ['train_loss', 'val_loss', 'test_loss', 'test_bleu', 'test_chrf', 'mine_bleu', 'mine_chrf', 'trainable_params', 'train_runtime_s', 'peak_gpu_gb'] if c in df]
    df = df[cols]

    df.to_csv('results_table.csv')
    display(df.round(3))

    metric_cols = [c for c in ['test_bleu', 'test_chrf', 'mine_bleu', 'mine_chrf'] if c in df]
    df[metric_cols].astype(float).plot.bar(figsize=(9, 4), rot=0, title='Translation scores by run')
    plt.ylabel('score'); plt.tight_layout()
    plt.savefig('compare_metrics.png', dpi=150, bbox_inches='tight'); plt.show()

    if hist:
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        for label, logs in hist.items():
            ax[0].plot(*_xy([(l['step'], l['loss']) for l in logs if 'loss' in l]), label=label)
            ax[1].plot(*_xy([(l['step'], l['eval_loss']) for l in logs if 'eval_loss' in l]), 'o-', label=label)
        ax[0].set_title('Train loss'); ax[1].set_title('Validation loss (per epoch)')
        for a in ax: a.set_xlabel('step'); a.legend()
        plt.tight_layout()
        plt.savefig('compare_loss.png', dpi=150, bbox_inches='tight'); plt.show()
    return df

# Evaluate the Un Fine Tuned Model for a Baseline
Scores the model before training, as the reference point for every experiment. Run once.

In [35]:
# Baseline run before fine-tuning
results = globals().get('results', {})
results['baseline'] = full_eval('baseline')
save_metrics('baseline', results['baseline'])

===== baseline =====


Training Loss,Validation Loss,Epoch,Train Loss
No log,No log,0,1.902143


Training Loss,Validation Loss,Epoch,Val Loss
No log,No log,0,1.951321


Training Loss,Validation Loss,Epoch,Test Loss
No log,No log,0,1.959916


train_loss 1.9021 | val_loss 1.9513 | test_loss 1.9599 | test perplexity 7.10


test:   0%|          | 0/29 [00:00<?, ?it/s]

test  | BLEU 11.34 | chrF 33.77


mine:   0%|          | 0/2 [00:00<?, ?it/s]

mine  | BLEU 10.89 | chrF 31.00
saved runs/baseline/metrics.json


# Fine-Tune with LoRA
Trains the adapters, and records training time and peak GPU memory.

In [ ]:
'''
About 603 steps per epoch, based on the numbers from filtered output.
Filtered train split has 96,455 rows, so N_TRAIN = round(96,455 × 10%) = 9,646.
One step uses 16 examples (batch 4 × accumulation 4).
Steps per epoch = 9,646 ÷ 16 = 602.9, rounded up to 603.
'''
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_out = trainer.train()
run_info = {
    'train_runtime_s': round(time.time() - t0),
    'peak_gpu_gb': round(torch.cuda.max_memory_allocated() / 1e9, 2),
    'trainable_params': sum(p.numel() for p in model.parameters() if p.requires_grad),
    'final_train_loss': train_out.metrics.get('train_loss'),
}
print(run_info)  # run the LoRA fine tuning loop, Traning loss loging step is 50

Epoch,Training Loss,Validation Loss


# Evaluate After Fine-Tuning
Scores the best checkpoint, then saves the metrics, loss curves and adapter for this run.

In [ ]:
# Evaluate After Fine Tuning
results[LABEL] = full_eval(LABEL)
save_metrics(LABEL, results[LABEL], run_info)
save_training(LABEL) # save LoRA adapter

##TensorBoard and Run Comparison
Opens TensorBoard and builds a table and charts that compare all finished experiments.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir .
compare_runs()

# Try out some examples and save the adapter
Translates a sample sentence with greedy and beam search decoding.

In [ ]:
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

def translate(la, beams=1, max_new_tokens=120):
    out = pipe(f"Latin: {la}\nEnglish:",
               max_new_tokens=max_new_tokens,
               do_sample=False,
               num_beams=beams,                  # 1 = greedy，4 = beam search
               repetition_penalty=1.1 if beams > 1 else 1.0,
               pad_token_id=tokenizer.eos_token_id,   # avoid "Setting pad_token_id" warning
               return_full_text=False)
    return out[0]['generated_text'].split('\n')[0].strip()

q = "Quo usque tandem abutere, Catilina, patientia nostra?"
print('greedy:', translate(q))
print('beam=4:', translate(q, beams=4))

# Reuse LORA
Reload a saved adapter on a fresh base model without retraining.
In case we need this

In [ ]:
'''base = AutoModelForCausalLM.from_pretrained(checkpoint, quantization_config=bnb, device_map='auto')
model = PeftModel.from_pretrained(base, f'runs/{LABEL}/adapter')'''